# MOT16 Faster R-CNN Training and Evaluation

This notebook follows the course-required approach:
- fine-tune a pretrained `Faster R-CNN`
- use data augmentation for stronger training images
- run a tracker on top of detector predictions
- generate videos with boxes and IDs
- evaluate tracking quality on held-out MOT16 training sequences

Main idea:
- the detector is trained using MOT16 pedestrian GT
- the tracker is used on top of detector outputs to maintain identities over time


## Workflow

1. Read MOT16 and create a pedestrian detection dataset.
2. Apply safe augmentations for object detection.
3. Fine-tune pretrained Faster R-CNN heads.
4. Validate the detector on held-out sequences.
5. Run a simple IoU-based tracker on detector outputs.
6. Save videos and MOTChallenge-format result files.
7. Evaluate results with your `MOTMetrics` wrapper.

Why this split?
- `MOT16/test` usually has no public GT, so use held-out sequences from `MOT16/train` for learning/evaluation.
- Later, run the final detector+tracker pipeline on `MOT16/test` for demo videos.


In [2]:
# Setup: imports, paths, device, and helper modules
from __future__ import annotations

import json
import os
import random
import shutil
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from torchvision import transforms
from IPython.display import Image as IPyImage, display

SEED = 21
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "code" else Path.cwd().resolve()
CODE_DIR = PROJECT_ROOT / "code"
MOT16_ROOT = PROJECT_ROOT / "MOT16"
OUTPUT_DIR = PROJECT_ROOT / "output"
FRCNN_DIR = OUTPUT_DIR / "fasterrcnn"
VIDEO_DIR = OUTPUT_DIR / "fasterrcnn_videos"
TRACK_RESULTS_DIR = OUTPUT_DIR / "fasterrcnn_tracking_results"
TORCH_HOME_DIR = PROJECT_ROOT / ".torch-cache"

for folder in [OUTPUT_DIR, FRCNN_DIR, VIDEO_DIR, TRACK_RESULTS_DIR, TORCH_HOME_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Store pretrained weights inside the project so Colab/local runs are reproducible.
os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from MOTMetrics import MOTMetrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", DEVICE)
print("TORCH_HOME:", os.environ["TORCH_HOME"])


PROJECT_ROOT: /Users/jainil/PycharmProjects/deep_learning_project
Device: mps
TORCH_HOME: /Users/jainil/PycharmProjects/deep_learning_project/.torch-cache


## Step 1 - Prepare MOT16 pedestrian detection data

We train Faster R-CNN as a **person detector**.

So for training we keep only GT rows with:
- `mark == 1`
- `class == 1`

We also split sequences into train/validation groups so the model is tested on unseen video sequences.


In [3]:
# MOT16 split and dataset indexing
GT_COLUMNS = [
    "frame", "id", "bb_left", "bb_top", "bb_width", "bb_height",
    "mark", "class", "visibility"
]

TRAIN_SEQUENCES = ["MOT16-02", "MOT16-04", "MOT16-05", "MOT16-09", "MOT16-10"]
VAL_SEQUENCES = ["MOT16-11", "MOT16-13"]
NUM_CLASSES = 2  # background + person


def load_sequence_gt(sequence_name: str) -> pd.DataFrame:
    """Load one MOT16 GT file and keep valid pedestrian rows only."""
    gt_path = MOT16_ROOT / "train" / sequence_name / "gt" / "gt.txt"
    gt_df = pd.read_csv(gt_path, header=None, names=GT_COLUMNS)
    gt_df = gt_df[(gt_df["mark"] == 1) & (gt_df["class"] == 1)].copy()
    gt_df["id"] = gt_df["id"].astype(int)
    return gt_df


sequence_stats = []
for seq_name in TRAIN_SEQUENCES + VAL_SEQUENCES:
    gt_df = load_sequence_gt(seq_name)
    image_count = len(list((MOT16_ROOT / "train" / seq_name / "img1").glob("*.jpg")))
    sequence_stats.append({
        "sequence": seq_name,
        "frames": image_count,
        "pedestrian_boxes": len(gt_df),
        "unique_ids": gt_df["id"].nunique(),
    })

display(pd.DataFrame(sequence_stats))


,sequence,frames,pedestrian_boxes,unique_ids
0,MOT16-02,600,17833,54
1,MOT16-04,1050,47557,83
2,MOT16-05,837,6818,125
3,MOT16-09,525,5257,25
4,MOT16-10,654,12318,54
5,MOT16-11,900,9174,69
6,MOT16-13,750,11450,107


## Step 2 - Data augmentation for detection

For detection, augmentations should improve robustness without breaking the boxes.

Good choices here:
- `ColorJitter`
- `GaussianBlur`
- `RandomGrayscale`
- horizontal flip with box update

These are safer than aggressive geometric transforms when you want a stable first training pipeline.


In [4]:
# Detection augmentations, dataset, and dataloaders
class DetectionTrainTransform:
    """Box-aware augmentations for Faster R-CNN training."""

    def __init__(self):
        self.color_aug = transforms.Compose([
            transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0)),
        ])
        self.gray_aug = transforms.RandomGrayscale(p=0.1)

    def __call__(self, image: Image.Image, target: dict):
        # Color-only augmentations do not change boxes.
        if random.random() < 0.7:
            image = self.color_aug(image)
        image = self.gray_aug(image)

        # Horizontal flip needs matching box updates.
        if random.random() < 0.5:
            image = F.hflip(image)
            width, _ = image.size
            boxes = target["boxes"].clone()
            if len(boxes) > 0:
                xmin = boxes[:, 0].clone()
                xmax = boxes[:, 2].clone()
                boxes[:, 0] = width - xmax
                boxes[:, 2] = width - xmin
                target["boxes"] = boxes

        image = F.to_tensor(image)
        return image, target


class DetectionEvalTransform:
    def __call__(self, image: Image.Image, target: dict):
        return F.to_tensor(image), target


class MOT16PedestrianDataset(Dataset):
    """Frame-level detection dataset built from MOT16 pedestrian GT."""

    def __init__(self, sequence_names, transform=None):
        self.sequence_names = sequence_names
        self.transform = transform
        self.samples = []

        for sequence_name in sequence_names:
            seq_dir = MOT16_ROOT / "train" / sequence_name
            img_dir = seq_dir / "img1"
            gt_df = load_sequence_gt(sequence_name)
            frame_groups = {frame: rows for frame, rows in gt_df.groupby("frame")}

            for image_path in sorted(img_dir.glob("*.jpg")):
                frame_number = int(image_path.stem)
                rows = frame_groups.get(frame_number)
                self.samples.append({
                    "sequence": sequence_name,
                    "frame": frame_number,
                    "image_path": image_path,
                    "rows": rows,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample["image_path"]).convert("RGB")
        rows = sample["rows"]

        boxes = []
        areas = []
        if rows is not None:
            for _, row in rows.iterrows():
                x1 = float(row["bb_left"])
                y1 = float(row["bb_top"])
                x2 = float(row["bb_left"] + row["bb_width"])
                y2 = float(row["bb_top"] + row["bb_height"])
                boxes.append([x1, y1, x2, y2])
                areas.append(float(row["bb_width"] * row["bb_height"]))

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels_tensor = torch.ones((len(boxes),), dtype=torch.int64) if boxes else torch.zeros((0,), dtype=torch.int64)
        areas_tensor = torch.tensor(areas, dtype=torch.float32) if areas else torch.zeros((0,), dtype=torch.float32)
        iscrowd_tensor = torch.zeros((len(boxes),), dtype=torch.int64) if boxes else torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": torch.tensor([index]),
            "area": areas_tensor,
            "iscrowd": iscrowd_tensor,
        }

        if self.transform is not None:
            image, target = self.transform(image, target)
        else:
            image = F.to_tensor(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = MOT16PedestrianDataset(TRAIN_SEQUENCES, transform=DetectionTrainTransform())
val_dataset = MOT16PedestrianDataset(VAL_SEQUENCES, transform=DetectionEvalTransform())

TRAIN_BATCH_SIZE = 2  # Safer for 4-6 GB VRAM.
VAL_BATCH_SIZE = 1

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Train frames:", len(train_dataset))
print("Val frames:", len(val_dataset))


Train frames: 3666
Val frames: 1650


## Step 3 - Build Faster R-CNN

This follows the pattern you were given:
- load pretrained `fasterrcnn_resnet50_fpn`
- freeze the backbone
- replace the prediction head
- optimize only trainable parameters


In [5]:
# Build Faster R-CNN and freeze the backbone
FRCNN_WEIGHTS = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9
NUM_EPOCHS = 10
RUN_NAME = f"fasterrcnn_heads_e{NUM_EPOCHS}"
BEST_MODEL_PATH = FRCNN_DIR / f"{RUN_NAME}_best.pth"

model = fasterrcnn_resnet50_fpn(weights=FRCNN_WEIGHTS)

# Freeze backbone layers so only the detection head is fine-tuned.
for param in model.backbone.parameters():
    param.requires_grad = False

# Replace the classifier head with the correct number of classes.
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

# Only optimize trainable parameters, matching the course instruction.
params_to_optimize = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params_to_optimize, lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print("Trainable parameter tensors:", len(params_to_optimize))
print("Best model path:", BEST_MODEL_PATH)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /Users/jainil/PycharmProjects/deep_learning_project/.torch-cache/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100.0%


Trainable parameter tensors: 14
Best model path: /Users/jainil/PycharmProjects/deep_learning_project/output/fasterrcnn/fasterrcnn_heads_e10_best.pth


## Step 4 - Train for a few epochs

The next cell runs a simple training loop and also computes validation loss.

Why validation loss?
- it is easy to compute with Faster R-CNN
- it helps you see whether the model is learning
- later, you still evaluate tracking with MOT metrics on held-out sequences


In [6]:
# Training and validation loops for Faster R-CNN

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

        loss_dict = model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()

    return running_loss / max(len(loader), 1)


def compute_validation_loss(model, loader, device):
    # Torchvision detection models return losses only in train mode.
    model.train()
    running_loss = 0.0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
            loss_dict = model(images, targets)
            total_loss = sum(loss for loss in loss_dict.values())
            running_loss += total_loss.item()

    return running_loss / max(len(loader), 1)


history = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = compute_validation_loss(model, val_loader, DEVICE)
    lr_scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

history_df = pd.DataFrame(history)
display(history_df)
print("Best validation loss:", best_val_loss)
print("Saved best model to:", BEST_MODEL_PATH)


: 

## Step 5 - Detector inference + tracker

This tracker is intentionally simple and commented so you can understand it.

It is an IoU-based tracker:
- detections in the next frame are matched to current tracks by IoU
- matched boxes keep the same ID
- unmatched detections start new IDs
- tracks survive for a few missed frames using a buffer

This is easier to study than a large external tracker and works well as a course project baseline.


In [ ]:
# Simple IoU tracker and sequence inference helpers
@dataclass
class TrackState:
    track_id: int
    bbox: np.ndarray
    score: float
    missed: int = 0


def compute_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


class SimpleIoUTracker:
    def __init__(self, iou_threshold=0.3, max_missed=20):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed
        self.tracks = []
        self.next_id = 1

    def update(self, detections):
        # detections: list of dicts with keys bbox and score
        if not self.tracks:
            for det in detections:
                self.tracks.append(TrackState(self.next_id, np.array(det["bbox"], dtype=float), det["score"], 0))
                self.next_id += 1
            return self.tracks

        cost_matrix = np.ones((len(self.tracks), len(detections)), dtype=float)
        for i, track in enumerate(self.tracks):
            for j, det in enumerate(detections):
                cost_matrix[i, j] = 1.0 - compute_iou(track.bbox, det["bbox"])

        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        matched_tracks = set()
        matched_dets = set()

        for track_idx, det_idx in zip(row_ind, col_ind):
            iou = 1.0 - cost_matrix[track_idx, det_idx]
            if iou >= self.iou_threshold:
                self.tracks[track_idx].bbox = np.array(detections[det_idx]["bbox"], dtype=float)
                self.tracks[track_idx].score = detections[det_idx]["score"]
                self.tracks[track_idx].missed = 0
                matched_tracks.add(track_idx)
                matched_dets.add(det_idx)

        for track_idx, track in enumerate(self.tracks):
            if track_idx not in matched_tracks:
                track.missed += 1

        self.tracks = [track for track in self.tracks if track.missed <= self.max_missed]

        for det_idx, det in enumerate(detections):
            if det_idx not in matched_dets:
                self.tracks.append(TrackState(self.next_id, np.array(det["bbox"], dtype=float), det["score"], 0))
                self.next_id += 1

        return self.tracks


def build_fasterrcnn_inference_model(weights_path: Path):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model


In [ ]:
# Run detector + tracker on one sequence and save a video plus MOT result file
INFER_SEQUENCE = VAL_SEQUENCES[0]  # Use a validation sequence first for evaluation.
INFER_SPLIT = "train"
INFER_SCORE_THRESH = 0.6
TRACK_IOU_THRESHOLD = 0.3
TRACK_MAX_MISSED = 20
EXPORT_MAX_FRAMES = 300  # Set to None to use all frames.

inference_model = build_fasterrcnn_inference_model(BEST_MODEL_PATH)
tracker = SimpleIoUTracker(iou_threshold=TRACK_IOU_THRESHOLD, max_missed=TRACK_MAX_MISSED)

seq_dir = MOT16_ROOT / INFER_SPLIT / INFER_SEQUENCE
img_dir = seq_dir / "img1"
seqinfo = configparser.ConfigParser()
seqinfo.read(seq_dir / "seqinfo.ini")
sequence_fps = seqinfo.getint("Sequence", "frameRate", fallback=30)

image_paths = sorted(img_dir.glob("*.jpg"))
if EXPORT_MAX_FRAMES is not None:
    image_paths = image_paths[:EXPORT_MAX_FRAMES]

video_frames = []
mot_lines = []

for frame_idx, image_path in enumerate(image_paths, start=1):
    pil_image = Image.open(image_path).convert("RGB")
    image_tensor = F.to_tensor(pil_image).to(DEVICE)

    with torch.no_grad():
        predictions = inference_model([image_tensor])[0]

    detections = []
    for box, score, label in zip(predictions["boxes"], predictions["scores"], predictions["labels"]):
        score_value = float(score.cpu())
        label_value = int(label.cpu())
        if score_value < INFER_SCORE_THRESH or label_value != 1:
            continue
        detections.append({
            "bbox": box.cpu().numpy().tolist(),
            "score": score_value,
        })

    active_tracks = tracker.update(detections)

    frame_np = np.array(pil_image)
    for track in active_tracks:
        if track.missed > 0:
            continue
        x1, y1, x2, y2 = track.bbox.astype(int)
        cv2.rectangle(frame_np, (x1, y1), (x2, y2), (255, 80, 0), 3)
        cv2.putText(frame_np, f"ID {track.track_id}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 80, 0), 2)
        width = float(x2 - x1)
        height = float(y2 - y1)
        mot_lines.append(f"{frame_idx},{track.track_id},{x1:.2f},{y1:.2f},{width:.2f},{height:.2f},{track.score:.6f},-1,-1,-1")

    video_frames.append(frame_np)

video_path = VIDEO_DIR / f"{INFER_SPLIT}_{INFER_SEQUENCE}_fasterrcnn_track.gif"
results_path = TRACK_RESULTS_DIR / f"{INFER_SEQUENCE}_fasterrcnn.txt"
imageio.mimsave(video_path, video_frames, fps=sequence_fps)
results_path.write_text("\n".join(mot_lines))

print("Saved video:", video_path)
print("Saved MOT results:", results_path)
display(IPyImage(filename=str(video_path)))


## Step 6 - Evaluate tracking quality

This uses your `MOTMetrics` class, which depends on the MATLAB MOT devkit.

If MATLAB Engine is not installed yet, the cell will explain what is missing.


In [ ]:
# Evaluate the saved tracking file with MOTMetrics
metrics = MOTMetrics(seqName=INFER_SEQUENCE)
try:
    metrics.compute_metrics_per_sequence(
        sequence=INFER_SEQUENCE,
        pred_file=str(results_path),
        gt_file=str(MOT16_ROOT / INFER_SPLIT / INFER_SEQUENCE / "gt" / "gt.txt"),
        gtDataDir=str(MOT16_ROOT / INFER_SPLIT),
        benchmark_name="MOT16",
    )
    metrics.compute_clearmot()
    metrics_df = pd.DataFrame([metrics.to_dict()])
    display(metrics_df)
except Exception as exc:
    print("Evaluation could not run yet:", exc)
    print("Install MATLAB Engine for Python and make sure matlab_devkit/ is available.")
